In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import boto3
from io import BytesIO
from dotenv import load_dotenv

In [2]:
# 1. Load Konfigurasi
load_dotenv()

True

In [3]:
# 2. Setup Koneksi
# Postgres
db_url = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

In [4]:
# MinIO
s3_client = boto3.client(
    's3',
    endpoint_url=f"http://{os.getenv('MINIO_ENDPOINT')}",
    aws_access_key_id=os.getenv('MINIO_ACCESS_KEY'),
    aws_secret_access_key=os.getenv('MINIO_SECRET_KEY')
)

In [5]:
BUCKET_NAME = "datalake-kelompok2"

In [6]:
def upload_to_minio(df, object_name):
    csv_buffer = BytesIO()
    df.to_csv(csv_buffer, index=False)
    s3_client.put_object(Bucket=BUCKET_NAME, Key=object_name, Body=csv_buffer.getvalue())
    print(f"✅ Berhasil upload {object_name} ke MinIO!")

In [7]:
# perbaikan (by malika)
def upload_file_bytes(local_path, object_name, content_type=None):
    with open(local_path, "rb") as f:
        s3_client.put_object(
            Bucket=BUCKET_NAME,
            Key=object_name,
            Body=f.read(),
            ContentType=content_type or "application/octet-stream"
        )
    print(f"✅ Berhasil upload {object_name} ke MinIO!")

In [12]:
def main():
    # Pastikan bucket ada
    try:
        s3_client.create_bucket(Bucket=BUCKET_NAME)
    except:
        pass

    print("🚀 Memulai proses ingestion...")

    # A. Ambil data dari Postgres (SQL)
    print("Reading from Postgres...")
    df_sql = pd.read_sql("SELECT * FROM stock_move", engine)
    upload_to_minio(df_sql, "raw/stock_transactions.csv")

    # B. Ambil data dari CSV
    print("Reading from local CSV...")
    df_csv = pd.read_csv("data/raw/Grocery_Inventory.csv")
    upload_to_minio(df_csv, "raw/Grocery_Inventory.csv")

    # C. Ambil data dari JSON
    # print("Reading from local JSON...")
    # df_json = pd.read_json("data/raw/suppliers_info.json")
    # upload_to_minio(df_json, "raw/suppliers_info.json")

    # perbaikan (by malika)
    print("Reading from local JSON...")
    upload_file_bytes("data/raw/suppliers_info.json", "raw/suppliers_info.json", "application/json")

    print("\n✨ SEMUA PROSES INGESTION SELESAI!")

In [13]:
import os
os.chdir('/app')

In [14]:
if __name__ == "__main__":
    main()

🚀 Memulai proses ingestion...
Reading from Postgres...
✅ Berhasil upload raw/stock_transactions.csv ke MinIO!
Reading from local CSV...
✅ Berhasil upload raw/Grocery_Inventory.csv ke MinIO!
Reading from local JSON...
✅ Berhasil upload raw/suppliers_info.json ke MinIO!

✨ SEMUA PROSES INGESTION SELESAI!
